# Character-Level LSTM for Shakespeare Text Generation

## Cell 1 — Import Libraries

In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

## Cell 2 — Download Shakespeare Dataset

In [ ]:
path = tf.keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
)

text = open(path, "rb").read().decode("utf-8")

print("Dataset downloaded successfully!")
print("Total characters:", len(text))

## Cell 3 — View Dataset

In [ ]:
print(text[:500])

## Cell 4 — Create Character Vocabulary

In [ ]:
vocab = sorted(set(text))
vocab_size = len(vocab)

print("Vocabulary size:", vocab_size)
print("Characters:", vocab)

## Cell 5 — Convert Characters to Numbers

In [ ]:
char_to_id = {char: i for i, char in enumerate(vocab)}
id_to_char = np.array(vocab)

encoded_text = np.array(
    [char_to_id[c] for c in text],
    dtype=np.int32
)

print("Original text:")
print(text[:50])
print("\nEncoded text:")
print(encoded_text[:50])

## Cell 6 — Create Training Sequences

In [ ]:
seq_length = 100

dataset = tf.data.Dataset.from_tensor_slices(encoded_text)

dataset = dataset.batch(
    seq_length + 1,
    drop_remainder=True
)

dataset = dataset.map(
    lambda x: (x[:-1], x[1:])
)

print("Sequences created successfully!")

## Cell 7 — Shuffle and Batch

In [ ]:
BATCH_SIZE = 64

dataset = dataset.shuffle(
    10000
).batch(
    BATCH_SIZE,
    drop_remainder=True
)

print("Dataset ready for training!")

## Cell 8 — Build LSTM Model

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=64
    ),
    tf.keras.layers.LSTM(
        128,
        return_sequences=True
    ),
    tf.keras.layers.Dense(
        vocab_size
    )
])

model.summary()

## Cell 9 — Compile Model

In [ ]:
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    )
)

print("Model compiled successfully!")

## Cell 10 — Train the Model

In [ ]:
EPOCHS = 5

history = model.fit(
    dataset,
    epochs=EPOCHS
)

## Cell 11 — Plot Training Loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    history.history["loss"],
    marker="o"
)
plt.title("Character-Level LSTM Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

## Cell 12 — Text Generation Function

In [ ]:
def generate_text(start_text, length=300):

    generated = start_text

    for _ in range(length):

        input_text = generated[-seq_length:]

        input_ids = [
            char_to_id[c]
            for c in input_text
        ]

        input_ids = np.array(input_ids).reshape(1, -1)

        predictions = model.predict(
            input_ids,
            verbose=0
        )

        predictions = predictions[:, -1, :]

        predicted_id = tf.random.categorical(
            predictions,
            num_samples=1
        )[0, 0].numpy()

        generated += id_to_char[predicted_id]

    return generated

## Cell 13 — Generate Shakespeare-Like Text

In [ ]:
generated_text = generate_text(
    "ROMEO: ",
    300
)

print("========== GENERATED TEXT ==========")
print(generated_text)